In [ ]:
import pandas as pd
import numpy as np
import os
from joblib import dump, load
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import OneHotEncoder
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, cross_validate
from collections import Counter
from sklearn.metrics import accuracy_score
from pprint import pprint
import random
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

dados = pd.read_csv ("Databases/PDACCTRL_samplesXfeatures_RAW.csv", sep=',')
dados

In [ ]:
#normalizar
dados_num = dados.drop(columns=["Sample_ID", "Grupo", "y"])
dados_cat =dados[["Grupo"]]

#Instaciar o objeto normalizador
normalizador = preprocessing.MinMaxScaler()
#treinar o modelo normalizador
modelo_normalizador_num = normalizador.fit(dados_num)
#Salvando modelo normalizador para uso posterior
os.makedirs('models_Tree_GPU', exist_ok=True)
dump(modelo_normalizador_num, open("models_Tree_GPU/normalizador_TCC_Tree_GPU.model" , "wb" ))

#normalizar os atributos numericos
dados_num_normalizado = modelo_normalizador_num.transform(dados_num)
# Normalizar os dados Categoricos
dados_cat_normalizado = pd.get_dummies(dados_cat, dtype='int', dummy_na=True)
# Converter dados numéricos normalizados para DataFrame
dados_num_df = pd.DataFrame(dados_num_normalizado, columns=dados_num.columns)
# unir atributos num e cat que foram normalizados
dados_final = dados_num_df.join(dados_cat_normalizado)
dados_final

In [ ]:
# remover a classe 'Grupo_nan' por ser insignificante
#dados_filtered = dados[~dados['Management'].isin(['missing', 'simultaneous appendectomy'])].copy()
dados_filtered = dados_final.drop(columns=['Grupo_nan'])

# Segmentar os dados em atributos e classes
dados_classes = dados_filtered[['Grupo_Controle', 'Grupo_PDAC']]
dados_atributos = dados_filtered.drop(columns=['Grupo_Controle', 'Grupo_PDAC'])

# Preprocessamento
# Tratar colunas categóricas
dados_cat = dados_atributos.select_dtypes(include=['object']).columns
dados_cat_normalizado = pd.get_dummies(dados_cat, dtype='int')

# Tratar colunas numéricas
dados_num = dados_atributos.select_dtypes(include=['number']).columns

# Preencher NaNs numéricos com a média da coluna
dados_num_preenchido = dados_atributos[dados_num].fillna(dados_atributos[dados_num].mean())

# Juntar numéricos preenchidos e categóricos dummificados
dados_processados = pd.concat([dados_num_preenchido, dados_cat_normalizado], axis=1)

# Criar uma coluna de classe unificada (1 = PDAC, 0 = Controle)
# Isso transforma o DataFrame de 2 colunas em uma única Series binária
dados_classes_bin = np.where(dados_classes['Grupo_PDAC'] == 1, 1, 0)

# Construir o objeto SMOTE e executar o fit_resample
resampler = SMOTE(random_state=42)
dados_atributos_b, dados_classes_b = resampler.fit_resample(dados_processados, dados_classes_bin)

# Verificar a frequência das classes balanceadas
print('#### FREQUÊNCIA DAS CLASSES APÓS O BALANCEAMENTO ####')
print(Counter(dados_classes_b))

In [ ]:
# Dicionario para armazenar os dados balanceados das classes
dados_balanceados = {}

print("Início do balanceamento")

# Loop para balancear cada classe (3x)
for classe in dados_classes.columns:
    print(f"\nBalanceando para a classe: '{classe}'")

    # Criar vetor binario para a classe atual: 1 se for essa classe, 0 caso contrário
    y_bin = dados_classes[classe]

    # Instanciar SMOTE com random_state para reprodutibilidade
    smote = SMOTE(random_state=42)

    # Aplicar o balanceamento com SMOTE
    X_res, y_res = smote.fit_resample(dados_processados, y_bin)

    # Exibir a contagem das classes após balanceamento
    print("Distribuição das classes após balanceamento:")
    print(Counter(y_res))

    # Armazenar os dados balanceados para essa classe
    dados_balanceados[classe] = (X_res, y_res)

In [ ]:
# Treinar RandomForest por classe e calcular permutation importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

rf_models = {}
rf_importances = {}
rf_perm = {}

for classe, (X, y) in dados_balanceados.items():
    print(f"\nTreinamento com RandomForest para: {classe}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    rf = RandomForestClassifier(n_estimators=50, max_features='sqrt', random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    # salvar
    nome_rf = f"modelo_RF_{classe.replace(' ', '_')}.joblib"
    caminho_rf = os.path.join(os.getcwd(), 'models_Tree', nome_rf)
    dump(rf, caminho_rf)
    print(f"RF salvo em: {caminho_rf}")

    # feature importances
    imp = rf.feature_importances_
    rf_models[classe] = rf
    rf_importances[classe] = imp

    # permutation importance (mais robusto)
    print('Calculando permutation importance (pode demorar) ...')
    perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
    rf_perm[classe] = perm

    # mostrar top20 por impurity e por permutation
    cols = pd.Index(dados_processados.columns)
    imp_df = pd.DataFrame({'feature': cols, 'impurity_imp': imp}).sort_values('impurity_imp', ascending=False)
    perm_means = pd.Series(perm.importances_mean, index=cols)
    perm_df = perm_means.sort_values(ascending=False).rename('perm_mean')

    print('\nTop 20 (impurity)')
    print(imp_df.head(20))
    print('\nTop 20 (permutation)')
    print(perm_df.head(20))


print('\nRandomForest training + permutation importance completo.')